In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default='notebook'

import tensorflow as tf
from tensorboard import notebook
%load_ext tensorboard

In [2]:
tf.__version__

'2.1.0'

## Загрузка данных

In [3]:
train_df = pd.read_csv('datas/project/fashion-mnist_train.csv')
test_df = pd.read_csv('datas/project/fashion-mnist_test.csv')

In [4]:
# разбитие по переменным 
Y_train, X_train = train_df.iloc[:,0], train_df.iloc[:,1:]
y_test, x_test = test_df.iloc[:,0], test_df.iloc[:,1:]
# нормализация признаков
X_train = tf.keras.utils.normalize(X_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)
# выделение из тренировочных данных валидационной выборки (random_state=42 для повторения результата)
x_train, x_val, y_train, y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)
# представление ответов в One-Hot Encoding
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
y_val = tf.keras.utils.to_categorical(y_val)

In [5]:
y_train.shape, x_train.shape

((48000, 10), (48000, 784))

In [6]:
def get_pict(position=False): # возвращает фотографию из x_train
    if not position: position = np.random.randint(0, len(x_train)-1)
    return np.flip(x_train.loc[x_train.index[position],:].values.reshape(28,28))

In [29]:
fig = make_subplots(cols=3, shared_yaxes=True)

fig.add_trace(go.Heatmap(z=get_pict(), 
                         colorscale=['white', 'black'], 
                         colorbar={'showticklabels': False,
                                   'thickness': 1,}), 1, 1)
fig.add_trace(go.Heatmap(z=get_pict(), 
                         colorscale=['white', 'black'], 
                         colorbar={'showticklabels': False,
                                   'thickness': 1,}), 1, 2)
fig.add_trace(go.Heatmap(z=get_pict(), 
                         colorscale=['white', 'black'], 
                         colorbar={'showticklabels': False,
                                   'thickness': 1,}), 1, 3)

layout = {'title': {'text': 'Пример входных данных',  # Заголовок графика
                    'x': 0.5,  # позиционирование title по центру
                    },
          'hovermode': False,
          'height': 400
          }

fig.update_layout(layout)
fig.show()

## Логистическая регрессия

Для решения задачи классификации предлагается начать с использования логистической регрессии. В данном случае, количество признаков равно 28x28=784, так же мы имеем 60000 объектов в тренировочной выборке. Поэтому рекомендуется использовать tensorflow или keras для выполнения этого задания. Используйте стохастический градиентный спуск (stochastic gradient descent) в качестве алгоритма оптимизации.

По своей сути, логистическая регрессия может быть реализована как нейронная сеть без скрытых слоев. В выходном слое содержится количество нейронов, равное количеству классов. В качестве функции активации выходного слоя следует использовать softmax.

Обучите логистическую регрессию на тренировочной выборке и оцените качество на тестовой выборке используя метрику accuracy. Постройте график качества модели на валидационной выборке от количества эпох. Для этого вы можете использовать утилиту tensorboard.
<ul>
    <li><a href="https://www.tensorflow.org/guide/summaries_and_tensorboard" target="_blank">tensorboard в tensorflow</a></li>
    <li><a href="https://keras.io/callbacks/#tensorboard" target="_blank">tensorboard в keras</a></li>
</ul>

In [10]:
log_reg_path = 'datas/project/logs/log_reg' # папка для хранения данных для tensorflow

# функция построения модели логистической регрессии
def log_reg(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(10, activation='softmax', input_shape=(784,)),
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='sgd',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = log_reg_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=(x_val, y_val),
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [11]:
model_log_reg = log_reg(x_train, y_train, epochs=10, batch_size=1000)
_, accuracy_log_reg = model_log_reg.evaluate(x_test, y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 10s 198us/sample - loss: 2.3040 - accuracy: 0.0896 - val_loss: 2.2999 - val_accuracy: 0.1032
Epoch 2/10
48000/48000 [==============================] - 1s 19us/sample - loss: 2.2963 - accuracy: 0.1193 - val_loss: 2.2921 - val_accuracy: 0.1358
Epoch 3/10
48000/48000 [==============================] - 1s 16us/sample - loss: 2.2886 - accuracy: 0.1548 - val_loss: 2.2844 - val_accuracy: 0.1791
Epoch 4/10
48000/48000 [==============================] - 1s 16us/sample - loss: 2.2810 - accuracy: 0.1933 - val_loss: 2.2768 - val_accuracy: 0.2202
Epoch 5/10
48000/48000 [==============================] - 1s 19us/sample - loss: 2.2734 - accuracy: 0.2362 - val_loss: 2.2692 - val_accuracy: 0.2634
Epoch 6/10
48000/48000 [==============================] - 1s 17us/sample - loss: 2.2659 - accuracy: 0.2791 - val_loss: 2.2617 - val_accuracy: 0.3044
Epoch 7/10
48000/48000 [==============================

In [12]:
pd.DataFrame(model_log_reg.history.history) # фрейм всех метрик качества при обучении модели

,loss,accuracy,val_loss,val_accuracy
0,2.304031,0.089625,2.299891,0.103167
1,2.296280,0.119313,2.292117,0.135833
2,2.288598,0.154792,2.284414,0.179083
3,2.280985,0.193312,2.276782,0.220250
4,2.273435,0.236167,2.269215,0.263417
5,2.265949,0.279063,2.261712,0.304417
6,2.258521,0.319813,2.254271,0.340833
7,2.251153,0.353667,2.246890,0.374333
8,2.243842,0.386229,2.239567,0.406500
9,2.236585,0.420083,2.232301,0.443250


In [13]:
print('Точность модели логистической регрессии на валидационной выборке - {:.2f} %'.format(accuracy_log_reg*100))

accuracy = model_log_reg.history.history['accuracy']

log_reg_graph = go.Scatter(x=np.arange(len(accuracy)),
                           y=accuracy*100,
                           name='Логистическая регрессия',
)

layout = {'title': {'text': 'График качества модели логистической регрессии', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=log_reg_graph,layout=layout).show()

Точность модели логистической регрессии на валидационной выборке - 44.34 %


## Полносвязная нейронная сеть

Далее, попробуйте реализовать полносвязную нейронную сеть с несколькими скрытыми слоями. Обучите модель и посчитайте качество на тестовой выборке. Как оно изменилось в сравнении с логистической регрессией? Как вы можете объяснить этот результат?

In [14]:
fcnn_path = 'datas/project/logs/fcnn'

# функция построения модели полносвязной нейронной сети
def fcnn(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(784, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='Adam',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = fcnn_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=(x_val, y_val),
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [15]:
model_fcnn = fcnn(x_train, y_train, epochs=10, batch_size=1000)
_, accuracy_fcnn = model_fcnn.evaluate(x_test, y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 6s 119us/sample - loss: 1.2495 - accuracy: 0.6687 - val_loss: 0.7167 - val_accuracy: 0.7502
Epoch 2/10
48000/48000 [==============================] - 5s 95us/sample - loss: 0.6244 - accuracy: 0.7807 - val_loss: 0.5662 - val_accuracy: 0.7967
Epoch 3/10
48000/48000 [==============================] - 5s 97us/sample - loss: 0.5256 - accuracy: 0.8162 - val_loss: 0.5051 - val_accuracy: 0.8205
Epoch 4/10
48000/48000 [==============================] - 4s 92us/sample - loss: 0.4829 - accuracy: 0.8291 - val_loss: 0.4778 - val_accuracy: 0.8282
Epoch 5/10
48000/48000 [==============================] - 4s 93us/sample - loss: 0.4575 - accuracy: 0.8381 - val_loss: 0.4535 - val_accuracy: 0.8405
Epoch 6/10
48000/48000 [==============================] - 4s 92us/sample - loss: 0.4356 - accuracy: 0.8456 - val_loss: 0.4392 - val_accuracy: 0.8446
Epoch 7/10
48000/48000 [==============================]

In [17]:
print('Точность модели полносвязной нейронной сети на валидационной выборке - {:.2f} %'.format(accuracy_fcnn*100))

accuracy = model_fcnn.history.history['accuracy']

fcnn_graph = go.Scatter(x=np.arange(len(accuracy)),
                        y=accuracy*100,
                        name='Полносвязная нейронная сеть',
)

layout = {'title': {'text': 'График качества модели полносвязной нейронной сети', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=fcnn_graph,layout=layout).show()

Точность модели полносвязной нейронной сети на валидационной выборке - 86.17 %


Данная модель показала лучший результат точности, чем предыдущая. Полносвязная нейронка хороша в анализе изображений

## Сверточная нейронная сеть

После этого вам предлагается реализовать сверточную нейронную сеть. В данном случае лучше использовать готовые слои, которые предоставляют keras или tensorflow.

Начните с модели с несколькими сверточными слоями. Так же рекомендуется использовать слои суб-дискретизации, например Max Pooling слои. Они понижают размерность сходных данных и выделяют наиболее важные признаки из данных. Посчитайте качество получившейся модели на тестовой выборке. Сравните полученные результаты с результатами полносвязной нейронной сети.

In [18]:
cnn1_path = 'datas/project/logs/cnn1/'

# функция построения модели сверточной нейронной сети
def cnn1(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Convolution2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='Adam',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = cnn1_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=(x_val.values.reshape(-1, 28, 28, 1), y_val),
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [19]:
model_cnn1 = cnn1(x_train.values.reshape(-1, 28, 28, 1), y_train, epochs=10, batch_size=1000)
_, accuracy_cnn1 = model_cnn1.evaluate(x_test.values.reshape(-1, 28, 28, 1), y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 41s 863us/sample - loss: 1.6464 - accuracy: 0.4534 - val_loss: 0.8427 - val_accuracy: 0.6992
Epoch 2/10
48000/48000 [==============================] - 39s 812us/sample - loss: 0.7331 - accuracy: 0.7317 - val_loss: 0.6779 - val_accuracy: 0.7467
Epoch 3/10
48000/48000 [==============================] - 38s 797us/sample - loss: 0.6340 - accuracy: 0.7637 - val_loss: 0.6018 - val_accuracy: 0.7757
Epoch 4/10
48000/48000 [==============================] - 38s 799us/sample - loss: 0.5805 - accuracy: 0.7838 - val_loss: 0.5740 - val_accuracy: 0.7792
Epoch 5/10
48000/48000 [==============================] - 39s 809us/sample - loss: 0.5447 - accuracy: 0.7964 - val_loss: 0.5373 - val_accuracy: 0.7985
Epoch 6/10
48000/48000 [==============================] - 38s 797us/sample - loss: 0.5219 - accuracy: 0.8060 - val_loss: 0.5063 - val_accuracy: 0.8073
Epoch 7/10
48000/48000 [====================

In [20]:
print('Точность модели сверточной нейронной сети на валидационной выборке - {:.2f} %'.format(accuracy_cnn1*100))
accuracy = model_cnn1.history.history['accuracy']

cnn1_graph = go.Scatter(x=np.arange(len(accuracy)),
                        y=accuracy,
                        name='Сверточная нейронная сеть',
)

layout = {'title': {'text': 'График качества модели сверточной нейронной сети', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=cnn1_graph,layout=layout).show()

Точность модели сверточной нейронной сети на валидационной выборке - 83.42 %


По сравнению с предыдущими моделями при неизменных значених кол-ва эпох и партий <i>batch_size</i> показазала <b>хороший результат</b>. Но качество полносвязной модели все равно лучше
<hr>

Далее, попробуйте увеличить количество слоев в вашей нейронной сети. Достаточно добавить несколько новых сверточных слоев. Проанализируете, как изменилось качество в этом случае.

В заключение, рекомендуется попробовать добавить Batch Normalization слои. Обычно они располагаются после сверточных слоев или слоев полносвязной нейронной сети. Обычно они улучшают качество модели, этим объясняется их популярность использования в современных архитектурах нейронных сетей. Однако, это требует проверки для конкретной модели и конкретного набора данных.

In [22]:
cnn2_path = 'datas/project/logs/cnn2/'
# функция построения модели полносвязной нейронной сети
def cnn2(x_train, y_train, epochs, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Convolution2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(64, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.Convolution2D(128, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer='Adam',
        metrics=['accuracy'],
    )

    logs= [tf.keras.callbacks.TensorBoard(
        log_dir = cnn2_path,
        write_graph=True,
        write_images=True,
        histogram_freq=1,
        profile_batch=100000000,
    )]
    
    model.fit(
        x=x_train,
        y=y_train,
        validation_data=(x_val.values.reshape(-1, 28, 28, 1), y_val),
        epochs = epochs,
        batch_size = batch_size,
        callbacks = logs,
        use_multiprocessing=True,
    )
    return model

In [23]:
model_cnn2 = cnn2(x_train.values.reshape(-1, 28, 28, 1), y_train, epochs=10, batch_size=1000)
_, accuracy_cnn2 = model_cnn2.evaluate(x_test.values.reshape(-1, 28, 28, 1), y_test)

Train on 48000 samples, validate on 12000 samples
Epoch 1/10
48000/48000 [==============================] - 48s 1ms/sample - loss: 1.1142 - accuracy: 0.6224 - val_loss: 2.0080 - val_accuracy: 0.2442
Epoch 2/10
48000/48000 [==============================] - 43s 887us/sample - loss: 0.5536 - accuracy: 0.7964 - val_loss: 1.7860 - val_accuracy: 0.5043
Epoch 3/10
48000/48000 [==============================] - 42s 885us/sample - loss: 0.4674 - accuracy: 0.8318 - val_loss: 1.5925 - val_accuracy: 0.7117
Epoch 4/10
48000/48000 [==============================] - 43s 903us/sample - loss: 0.4184 - accuracy: 0.8495 - val_loss: 1.4172 - val_accuracy: 0.8047
Epoch 5/10
48000/48000 [==============================] - 42s 884us/sample - loss: 0.3878 - accuracy: 0.8597 - val_loss: 1.1896 - val_accuracy: 0.8278
Epoch 6/10
48000/48000 [==============================] - 43s 885us/sample - loss: 0.3621 - accuracy: 0.8685 - val_loss: 1.0293 - val_accuracy: 0.8560
Epoch 7/10
48000/48000 [======================

In [24]:
print('Точность модели сверточной нейронной сети с большим кол-во скрытых слоев на валидационной выборке - {:.2f} %'.format(accuracy_cnn2*100))

accuracy = model_cnn2.history.history['accuracy']

cnn2_graph = go.Scatter(x=np.arange(len(accuracy)),
                        y=accuracy,
                        name='Сверточная нейронная сеть',
)

layout = {'title': {'text': 'График качества модели сверточной нейронной сети с большим кол-во скрытых слоев', # Заголовок графика
                    'x':0.5, # позиционирование title по центру
                   },
          'xaxis_title': 'Колличество эпох', # подпись Ох 
          'yaxis_title': 'Точность', # подпись Оу
         }

go.Figure(data=cnn2_graph,layout=layout).show()

Точность модели сверточной нейронной сети с большим кол-во скрытых слоев на валидационной выборке - 84.17 %


При увеличении кол-ва скрытых слоев наблюдается положительная динамика роста точности, хоть и незначетельная.

## Ответ

В качестве решения приложите архив, содержащий файл решения и все используемые для его работы файлы.Постройте график качества модели на валидационной выборке от количества эпох. Для этого вы можете использовать утилиту tensorboard.

In [ ]:
# проверка данных с помощью tensorboard
common_path = 'datas/project/logs/'
os.makedirs(common_path, exist_ok=True)

%tensorboard --logdir=$common_path --host localhost --port=6007

In [22]:
print('Точность моделей, проверенная на тестовых данных:\n\tЛогистическая регрессия: {:.2f} %;\n\tПолносвязная нейронная сеть: {:.2f} %;\n\tСверточная нейронная сеть: {:.2f} %;\n\tСверточная нейронная сеть с большим кол-во скрытых слоев: {:.2f} %'.format(
    accuracy_log_reg*100, accuracy_fcnn*100, accuracy_cnn1*100, accuracy_cnn2*100))

Точность моделей, проверенная на тестовых данных:
	Логистическая регрессия: 37.81 %;
	Полносвязная нейронная сеть: 58.30 %;
	Сверточная нейронная сеть: 26.98 %;
	Сверточная нейронная сеть с большим кол-во скрытых слоев: 71.88 %


In [25]:
print('Точность моделей, проверенная на тестовых данных:\n\tЛогистическая регрессия: {:.2f} %;\n\tПолносвязная нейронная сеть: {:.2f} %;\n\tСверточная нейронная сеть: {:.2f} %;\n\tСверточная нейронная сеть с большим кол-во скрытых слоев: {:.2f} %'.format(
    accuracy_log_reg*100, accuracy_fcnn*100, accuracy_cnn1*100, accuracy_cnn2*100))

Точность моделей, проверенная на тестовых данных:
	Логистическая регрессия: 44.34 %;
	Полносвязная нейронная сеть: 86.17 %;
	Сверточная нейронная сеть: 83.42 %;
	Сверточная нейронная сеть с большим кол-во скрытых слоев: 84.17 %


Каждая модель обучалась дольше предыдущей.<br> Самый лучший результат показала модель <b>cверточной нейронной сети с большим кол-вом скрытых слоев</b>, самый худший результат показала модель <b>cверточной нейронной сети c 2мя сверточными слоями</b>. Это доказывает, что самая <i>сложная модель не всегда приводит к лучшему результату</i>, но <i>при увеличении кол-ва слоев, качество модели увеличивается</i>.